In [11]:
import pymupdf
from pathlib import Path

In [12]:
pdf_path = Path("invoices_for_test/fatura_bradesco-8.pdf")
doc = pymupdf.open(pdf_path)

pdf_string = ""

for page in doc:
  pdf_string += f"\n{page.get_text()}"

print(pdf_string)


Vamos cuidar do nosso Planeta!
Com o Bradesco, você tem a opção de substituir sua fatura do Cartão em papel
por eletrônica. Bradesco Prime, lado a lado com a sua conveniência.
Acesse bradescoprime.com.br e cadastre-se no INFOEMAIL.
CTC CIDADE NOVA RJ TTO PL4
EDUARDO S G FELIX
CHARITAS
24370-015
NITEROI
RJ
Total da fatura
R$ 14.599,01
 Vencimento
10/10/2026
Previsão de fechamento da próxima fatura: 27/10/2026
Limite de compras
R$ 29.200,00
Limite de saque
R$ 4.380,00
Opções de pagamento
Recomendado
Pagamento total
R$ 14.599,01
Melhor opção
Sem juros  ou qualquer tipo de
encargo.
Parcelamento de Fatura
1ª parcela de R$ 1.514,87 + 12 de R$ 2.213,19
Para contratar esta opção é só pagar o valor da
1ª parcela do plano.
Juros de parcelamento:
12,50 % a.m
310,98% a.a
CET do parcelamento:
13,01 % a.m
334,13% a.a
Valor total financiado* :
100,00 %
R$ 13.397,50
Total a financiar (1):
97,66 %
R$ 13.084,14
Valor do IOF (2):
2,34 %
R$ 313,36
Valor total a pagar:
R$ 26.558,34
(*) O valor total finan

In [15]:
from functools import cached_property

from google.adk.agents import LlmAgent
from google.adk.models import Gemini
from google.genai import Client
import asyncio
import json
import re
from typing import Any, Dict, List, Optional, Tuple

from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
import google.auth
from google.cloud import storage
from google.genai import types
from google.adk.integrations.agent_registry import agent_registry as _agent_registry
from google.adk.integrations.agent_registry.agent_registry import AgentRegistry
from google.adk.tools import agent_tool
from google.adk.tools.google_search_tool import GoogleSearchTool
from google.adk.tools import url_context



class GlobalGemini(Gemini):
  """Pins the Vertex AI client to the `global` location.

  gemini-3 series models are only served from `global`; the default ADK
  `Gemini` integration constructs a `google.genai.Client` whose location
  defaults to the AgentEngine instance's region (e.g. `us-central1`) and
  fails with model-not-found for these models. Subclassing per the override
  pattern documented on `google.adk.models.google_llm.Gemini` lets the agent
  keep running in its regional AgentEngine instance while routing the model
  request to the global endpoint.
  """

  @cached_property
  def api_client(self) -> Client:
    return Client(vertexai=True, location="global")


# --- BEGIN: knowledge resources scaffolding (auto-generated) ---
# This block is emitted by the Lowcode Agent Designer code generator whenever the agent has
# attached knowledge resources. It loads each resource from its source GCS URI on every
# model call. Do not edit by hand: changes will be overwritten on the next download.

_KNOWLEDGE_RESOURCES: List[Dict[str, str]] = [
    {"fileName": 'Consolidado_Despesas__Distintos.csv', "gcsUri": 'gs://63660038188_us-west1_agent_studio_files/adk_artifacts/agent_1791557064633/knowledge_resources/user:Consolidado_Despesas__Distintos.csv/0', "mimeType": 'text/csv'},
]

_KNOWLEDGE_GCS_URI_RE = re.compile(r"^gs://([^/]+)/(.+)$")


def _parse_knowledge_gcs_uri(uri: str) -> Tuple[str, str]:
  match = _KNOWLEDGE_GCS_URI_RE.match(uri)
  if not match:
    raise ValueError(f"Invalid GCS URI: {uri}")
  return match.group(1), match.group(2)


# Quota project is pinned to the agent's user project so GCS reads send
# `x-goog-user-project=<project>` and do not fail `serviceusage.services.use`
# against whatever project ADC happens to resolve.
_KNOWLEDGE_CREDENTIALS, _ = google.auth.default(
    quota_project_id="63660038188",
)
_KNOWLEDGE_STORAGE_CLIENT = storage.Client(
    project="63660038188",
    credentials=_KNOWLEDGE_CREDENTIALS,
)


def _download_knowledge_blob(gcs_uri: str) -> bytes:
  bucket_name, blob_name = _parse_knowledge_gcs_uri(gcs_uri)
  blob = _KNOWLEDGE_STORAGE_CLIENT.bucket(bucket_name).blob(blob_name)
  return blob.download_as_bytes()


async def load_artifact_before_model(
    callback_context: CallbackContext,
    llm_request: LlmRequest,
) -> Optional[Any]:
  """Inlines every configured knowledge resource on every model call."""
  del callback_context  # unused; resources are static, not session-scoped.
  if not _KNOWLEDGE_RESOURCES:
    return None
  artifact_names = [kr["fileName"] for kr in _KNOWLEDGE_RESOURCES]
  llm_request.append_instructions([
      f"""You have a list of artifacts:
{json.dumps(artifact_names)}

When the user asks questions about any of the artifacts, you should use the
artifact data from request content directly.
"""
  ])
  blobs = await asyncio.gather(
      *(
          asyncio.to_thread(_download_knowledge_blob, kr["gcsUri"])
          for kr in _KNOWLEDGE_RESOURCES
      )
  )
  for kr, data in zip(_KNOWLEDGE_RESOURCES, blobs):
    mime_type = kr["mimeType"]
    if mime_type == "application/json":
      try:
        part = types.Part.from_text(text=data.decode("utf-8"))
      except UnicodeDecodeError:
        part = types.Part.from_bytes(data=data, mime_type=mime_type)
    else:
      part = types.Part.from_bytes(data=data, mime_type=mime_type)
    llm_request.contents.append(
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(text=f"Artifact {kr['fileName']} is:"),
                part,
            ],
        )
    )
  return None
# --- END: knowledge resources scaffolding ---
o_que_contas__google_search_agent = LlmAgent(
  name='O_que_contas__google_search_agent',
  model=GlobalGemini(model='gemini-3.5-flash'),
  description=(
      'Agent specialized in performing Google searches.'
  ),
  sub_agents=[],
  instruction='Use the GoogleSearchTool to find information on the web.',
  tools=[
    GoogleSearchTool()
  ],
  before_model_callback=load_artifact_before_model,
)
o_que_contas__url_context_agent = LlmAgent(
  name='O_que_contas__url_context_agent',
  model=GlobalGemini(model='gemini-3.5-flash'),
  description=(
      'Agent specialized in fetching content from URLs.'
  ),
  sub_agents=[],
  instruction='Use the UrlContextTool to retrieve content from provided URLs.',
  tools=[
    url_context
  ],
  before_model_callback=load_artifact_before_model,
)
root_agent = LlmAgent(
  name='O_que_contas_',
  model=GlobalGemini(model='gemini-3.5-flash'),
  description=(
      'Orchestrates different bank statements and invoice files into a single standardized CSV format.'
  ),
  sub_agents=[],
  instruction='You are a financial assistant that standardizes bank statements and invoices into a unified CSV format with the columns: data, merchant, valor pago, categoria, pagamento.\n\nStandard Schema:\n- Format: CSV table.\n- Columns: data (YYYY-MM-DD), merchant, valor pago (positive numeric values, \".\" as decimal separator), categoria, pagamento.\n\nFlow Selection (check in this order):\n1. Prompt starts with \'Pix XP\' -> Flow 4\n2. Prompt starts with \'Pix\' -> Flow 2\n3. Prompt starts with \'XP\' -> Flow 3\n4. Prompt starts with \'Bradesco Extrato\' -> Flow 5\n5. Content contains \'fatura_bradesco\' or is a Bradesco credit card invoice -> Flow 1\n6. User writes \'Concatenar\' -> Flow 6\n\nGlobal Categorization Rule (applies to ALL flows):\nFill the \"categoria\" column using this priority order:\n1. If the flow defines a fixed mapping (Flows 2, 4 and 5, and the \'Parcela\' rule in Flow 3), use that mapping.\n2. Otherwise, look up the merchant in the Knowledge file \'Consolidado_Despesas__Distintos.csv\'.\n   - Match ignoring case, accents, asterisks (*), store numbers (e.g. \'-0001\') and installment suffixes (e.g. \'01/12\').\n   - If a match is found, use EXACTLY the category from the CSV.\n3. Only if the merchant is not in the CSV, infer the category from the merchant name. Use Google Search only if the merchant type still cannot be identified.\n4. If still uncertain, use \'Outros\'.\n- Only use categories that already exist in \'Consolidado_Despesas__Distintos.csv\' or in the fixed mappings below. Never create new categories.\n- After the CSV table, list the merchants that were NOT found in the reference CSV, with the category you assigned, so the user can review them.\n\nFlow 1 - Fatura Bradesco (PDF containing \'fatura_bradesco\'):\n1. Extract data, merchant, and valor pago. Use the invoice due date to infer the year (YYYY-MM-DD). For international purchases, use the BRL amount.\n2. Set pagamento to \'Bradesco\'.\n3. Remove invoice payment rows (e.g. \'PAGTO. POR DEB EM C/C\', \'Pagamento de fatura\') and rows whose categoria is \'Doação\'.\n4. Classify categoria following the Global Categorization Rule.\n\nFlow 2 - Pix Bradesco (PDF with prompt starting with \'Pix\'):\n1. Extract data (take the year YYYY from the title string \'Bradesco DD_MM_YYYY\' and format as YYYY-MM-DD), merchant (from \'Referências\'), and valor pago. Skip the time column.\n2. Set pagamento to \'Pix\'.\n3. Keep only rows matching these references and assign the categoria:\n- \'Maria Anilde Alves Silva Frazão de Lima NU PAGAMENTOS - IP\' -> Faxina\n- \'MARTA MACIEL LEVY DE CARVALHO VIANNA BCO SANTANDER (BRASIL) S.A.\' -> Terapia\n- \'ENEL DISTRIBUICAO SAO PAULO ITAÚ UNIBANCO S.A.\' -> Luz\n- \'GABRIELLE MELO REGINATTO BANCO INTER\' -> Treino Rafael\n- \'BOC ODONTOLOGIA LIMITADA BCO C6 S.A.\' -> Dentista\n- \'LA BARBIERE BCO SANTANDER (BRASIL) S.A\' -> Corte cabelo\n- \'Patrícia Bertoni dos Reis NU PAGAMENTOS - IP\' -> Nutricionista\n- \'Claro CLARO PAY S.A. IP\' -> Internet\n- \'CAIXA DE ASSISTENCIA DOS FUNCIONARIOS DO BANCO DO BRASIL BCO DO BRASIL S.A.\' -> Plano de Saúde\n- \'FABIO KAZUO NAGASAWA BCO DO BRASIL S.A.\' -> Taekwondo\n\nFlow 3 - Cartão XP (CSV with prompt starting with \'XP\'):\n1. Treat the CSV delimiter as \';\'.\n2. Extract data (YYYY-MM-DD), merchant (from \'Estabelecimento\'), and valor pago. Skip \'Portador\'.\n3. Set pagamento to \'XP\'.\n4. Remove rows where \'Estabelecimento\' equals \'Pagamentos Validos Normais\'.\n5. If column \'Parcela\' is filled, set categoria to \'Assinatura\'. Otherwise classify following the Global Categorization Rule.\n\nFlow 4 - Pix XP (CSV with prompt starting with \'Pix XP\'):\n1. Extract data (YYYY-MM-DD), merchant, and valor pago (convert negative values to positive). Skip \'Hora\'.\n2. Set pagamento to \'Pix\'.\n3. Keep only rows where \'Descricao\' contains (case-insensitive): Claro, Multa, Seguradora, LELLO, MINISTERIO DA JUSTICA E SEGURANCA PUBLICA.\n4. Classify categoria as follows:\n- Claro -> Internet\n- Multa -> Multa\n- Seguradora -> Seguro auto\n- LELLO -> Condominio\n- MINISTERIO DA JUSTICA E SEGURANCA PUBLICA -> Passaporte\n\nFlow 5 - Extrato Bradesco (CSV with prompt starting with \'Bradesco Extrato\'):\n1. Extract data (YYYY-MM-DD), merchant (from \'Histórico\'), and valor pago (from \'Débito (R$)\'). Skip \'Dcto.\', \'Crédito (R$)\', and \'Saldo (R$)\'.\n2. Set pagamento to \'Boleto\'.\n3. Keep only rows where \'Histórico\' contains (case-insensitive): \'tv p/assinatura\' or \'Pagto Cobranca\'.\n4. Classify categoria as follows:\n- tv p/assinatura -> Internet\n- Pagto Cobranca -> Condominio\n\nFlow 6 - Concatenar:\nWhen the user writes \'Concatenar\', combine all processed rows from this session into a single consolidated CSV table. Direct integration with Google Sheets is not supported, so output the complete CSV table directly in your reply for the user to copy.',

  tools=[
    agent_tool.AgentTool(agent=o_que_contas__google_search_agent),
    agent_tool.AgentTool(agent=o_que_contas__url_context_agent),
    AgentRegistry(
      project_id='wagon-bootcamp-385919',
      location='global',
    ).get_mcp_toolset(
      mcp_server_name='projects/wagon-bootcamp-385919/locations/global/mcpServers/agentregistry-00000000-0000-0000-ed34-af0fb0397a5e',
    )
  ],
  before_model_callback=load_artifact_before_model,
)

ImportError: Missing required dependencies for Agent Identity Auth Manager. Please install with: pip install "google-adk[agent-identity]"

In [13]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP, USER = "o_que_contas", "eduardo"
session_service = InMemorySessionService()
runner = Runner(agent=root_agent, app_name=APP, session_service=session_service)
session = await session_service.create_session(app_name=APP, user_id=USER)

async def perguntar(texto, pdf_path=None):
    parts = [types.Part(text=texto)]
    if pdf_path:
        with open(pdf_path, "rb") as f:
            parts.append(types.Part.from_bytes(data=f.read(), mime_type="application/pdf"))
    msg = types.Content(role="user", parts=parts)
    async for ev in runner.run_async(user_id=USER, session_id=session.id, new_message=msg):
        if ev.is_final_response() and ev.content:
            return "".join(p.text or "" for p in ev.content.parts)

resp = await perguntar(pdf_string)
print(resp)

NameError: name 'root_agent' is not defined